# Fine-tuning and Deploying BERT

At this point, I've become impatient while I continue to tag local coverage of the KKK and other white terror organizations in 05_tag_local_report_x2.ipynb. I want to do some preliminary tests to see if my whole pipeline works! So, that's what I'm doing here–I'm seeing if I can use just my local coverage of lynchings (for now) plus my random stew to fine-tune BERT to classify texts as local coverage or not.

As I thought through this testing phase, I realized there were a couple ways I could review the results. One was the traditional way in machine learning–splitting my training data into a train/test split and seeing how well the model does. The other way was more practical in terms of my larger goals of identifying local and racially threatening reports in newspapers. It involved deploying my fine-tuned model to classify newspaper text and seeing what it labeled as local racially threatening coverage. With this approach, I was basically seeing if the model could identify sundown town content in a real-world example. It required some hand-keying of the results, but ultimately it would be testing the model as I hoped to do on a larger scale.

I decided to do both tests. As you'll see, both yielded promising results. There is still work to do to improve my model and incorporate other types of sundown town signals, but this notebook proved the concept, which is good. Here's how it went:

## 1) Prepare Test Data for BERT

As a first step, I decided to isolate an example of a newspaper from a county where a lynching occurred, but where my data sources didn't know the name of the victim. If you remember, in 02_identify_local_lynch_coverage.ipynb I paired digitized newspapers with local lynching cases, and if I knew the victim name, I mined those newspapers for references to them. In some cases, however, the victim name was unknown. I had the location, date, and other details of the case in data, but not the victim's name.

So, to test the whole process, my goal was to see if I could fine-tune a model to identify coverage of these local cases even when the victim's name was unknown. I reviewed my local_coverage_cases.csv and identified one example: a lynching that occurred in Alexander County, IL, on September 12th, 1913. If you take a look at that csv, you'll see the case referenced on rows 62 and 63. There were two victims. They were purportedly accused of assault and shot by a mob. This specifically occurred in the town of Tamms–just a little north of Cairo, IL. I had local newspapers digitized for the case–the Cairo Bulletin (sn93055779). The goal: could I fine-tune a BERT model to find local coverage about this case?

To start, I copied a version of the sn_code subdirectory (sn93055779). This subdirectory had all the ocr text of the Cairo Bulletin from years 1912 to 1915. Then I formatted this data into a csv file. This involved taking the details listed by the nested subdirectories (year, month, date, edition, sequence, etc.) and putting those values into csv columns as well as the full text of each page's ocr.

This resulted a csv with all the Cairo Bulletin ocr text–7,521 rows long, one row per newspaper page.

In [ ]:
from pathlib import Path
import csv
from transformers import AutoTokenizer

root = Path("test_data/sn93055779")
output_csv = Path("test_data/sn93055779.csv")

with output_csv.open("w", newline="", encoding="utf-8") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(["year", "month", "day", "ed", "seq", "text"])

    for ocr_file in root.rglob("ocr.txt"):
        rel = ocr_file.relative_to(root)
        parts = rel.parts

        # Expected layout: year/month/day/ed/seq/ocr.txt
        if len(parts) != 6:
            print(f"Skipping unexpected path: {rel}")
            continue

        year, month, day, ed, seq, _ = parts
        text = ocr_file.read_text(encoding="utf-8", errors="replace")

        writer.writerow([year, month, day, ed, seq, text])

print(f"Wrote {output_csv}")

Then I needed to tokenize the text. In particular, I needed to tokenize it with BERT so it would be compatible with my fine-tuning processes. BERT also has a max context length of 512 tokens, so I had to split all the ocr text up into 512 token chunks. These chunks are what would be classified as either "containing local coverage of racial violence" or not.

I also created a stride to ensure that I wasn't losing context between chunks. In other words, I made sure chunks contained overlapping tokens (64 tokens or 1/8th of their text) so the model would have an easier time capturing context.

This resulted a csv with all the Cairo Bulletin ocr text now prepared in classifiable chunks–90,431 rows long, one row per 512 token chunk of text.

In [ ]:
# tokenize with BERT tokenizer

input_csv = Path("test_data/sn93055779.csv")
output_csv = Path("test_data/sn93055779_BERT_ready.csv")

model_name = "bert-base-uncased"
max_length = 512          # total BERT input length, including special tokens
stride = 64               # overlap between neighboring windows

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Reserve space for special tokens like [CLS] and [SEP]
special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
window_size = max_length - special_tokens

if stride >= window_size:
    raise ValueError(f"stride must be smaller than window_size ({window_size})")

step = window_size - stride

with input_csv.open("r", newline="", encoding="utf-8") as f_in, \
     output_csv.open("w", newline="", encoding="utf-8") as f_out:

    reader = csv.DictReader(f_in)
    fieldnames = ["year", "month", "day", "ed", "seq", "chunk_index", "text"]
    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()

    for row_idx, row in enumerate(reader):
        text = row["text"] or ""

        # Tokenize without special tokens so we can control windowing directly
        token_ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)

        if not token_ids:
            continue

        chunk_index = 0
        for start in range(0, len(token_ids), step):
            end = start + window_size
            chunk_ids = token_ids[start:end]

            if not chunk_ids:
                continue

            chunk_text = tokenizer.decode(
                chunk_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )

            writer.writerow({
                "year": row["year"],
                "month": row["month"],
                "day": row["day"],
                "ed": row["ed"],
                "seq": row["seq"],
                "chunk_index": chunk_index,
                "text": chunk_text,
            })

            chunk_index += 1

            if end >= len(token_ids):
                break

print(f"Wrote {output_csv}")

## 2) Compile Training Data

With my example data in the Cairo Bulletin prepared, now I needed to prepare my fine-tuning (training) data for BERT. This was coming from two different places–the rows from my results_df_progress.csv where the "verification_status" column values were labeled as "verified" and the rows from my random_text_sample_500.csv where the "usability_status" column values were labeled as "usable". These rows contain both text of local coverage (in results_df_progress.csv) and random newspaper text that does not contain local coverage (in random_text_sample_500.csv). See notebooks 03_tag_local_reports.ipynb and 06_generate_random_stew.ipynb for more info on how I compiled this training set.

Anyway, I needed to put these two sources together into one training set, tokenize them, and segment them into appropriate context windows for BERT (512 tokens max). I started by pulling the appropriate subsets, labeling them for the model, and concatenating them:

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

# File paths
positive_path = "results_df_progress.csv"
negative_path = "random_text_sample_500.csv"
output_path = "test_data/bert_training_data.csv"

# Load data
pos_df = pd.read_csv(positive_path)
neg_df = pd.read_csv(negative_path)

# Select positive examples
pos_train = pos_df.loc[
    pos_df["verification_status"].astype(str).str.strip().str.lower() == "verified",
    ["verified_articles"]
].copy()
pos_train = pos_train.rename(columns={"verified_articles": "text"})
pos_train["label"] = 1
pos_train["source"] = "results_df_progress"

# Select negative examples
neg_train = neg_df.loc[
    neg_df["usability_status"].astype(str).str.strip().str.lower() == "usable",
    ["random_text"]
].copy()
neg_train = neg_train.rename(columns={"random_text": "text"})
neg_train["label"] = 0
neg_train["source"] = "random_text_sample_500"

# Combine
bert_df = pd.concat([pos_train, neg_train], ignore_index=True)

# Clean text
bert_df["text"] = bert_df["text"].fillna("").astype(str).str.strip()
bert_df = bert_df[bert_df["text"] != ""]

# Shuffle rows for training
bert_df = bert_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save
bert_df.to_csv(output_path, index=False)

Then I tokenized them. This resulted in 1,976 chunks for BERT's fine-tuning.

In [ ]:
input_path = "test_data/bert_training_data.csv"
output_path = "test_data/bert_training_data.csv"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

max_length = 512
stride = 50

df = pd.read_csv(input_path)

rows = []

for row_idx, row in df.iterrows():
    text = str(row["text"]).strip()
    label = row["label"]
    source = row.get("source", "")

    if not text:
        continue

    encoded = tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        stride=stride,
        return_overflowing_tokens=True,
        padding=False,
    )

    for chunk_id, input_ids in enumerate(encoded["input_ids"]):
        chunk_text = tokenizer.decode(
            input_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        ).strip()

        rows.append({
            "original_row": row_idx,
            "chunk_id": chunk_id,
            "text": chunk_text,
            "label": label,
            "source": source
        })

chunked_df = pd.DataFrame(rows)
chunked_df.to_csv(output_path, index=False)
print(f"Saved {len(chunked_df)} chunks to {output_path}")

## 3) Test BERT for Classification Ability

At this point, I could have technically fine-tuned and deployed BERT to classify my Cairo Bulletin data, but first I decided to do the ole' machine learning standard of testing my training data on itself, basically. This involved splitting the training data into an 80% training/20% testing data split. I went with BERT-base as well, the tried and true BERT model. Codex wrote the code here, I just reviewed it carefully and double-checked hyperparameters, Torch backend configurations for my MacBook, and then let it rip!

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from tqdm.auto import tqdm


# ----------------------------
# Settings
# ----------------------------

MODEL_NAME = "bert-base-uncased"

# Use bert_training_data_chunked.csv here if you already chunked long training texts.
TRAIN_PATH = "test_data/bert_training_data.csv"

TEST_PATH = "test_data/sn93055779_BERT_ready.csv"
TEMP_TEST_PATH = "test_data/sn93055779_BERT_ready.tmp.csv"

MODEL_OUTPUT_DIR = "bert_models/bert_local_violence_model"

BATCH_SIZE = 16
MAX_LENGTH = 512
RANDOM_STATE = 42
READ_CHUNK_SIZE = 2_000


# ----------------------------
# Device
# ----------------------------

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


# ----------------------------
# Dataset
# ----------------------------

class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(int(self.labels[idx]), dtype=torch.long),
        }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


# ----------------------------
# Load training data
# ----------------------------

df = pd.read_csv(TRAIN_PATH)

df = df[["text", "label"]].copy()
df["text"] = df["text"].fillna("").astype(str).str.strip()
df = df[df["text"] != ""]
df["label"] = df["label"].astype(int)

print("Training label counts:")
print(df["label"].value_counts())

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["label"],
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = TextClassificationDataset(
    train_df["text"],
    train_df["label"],
    tokenizer,
    max_length=MAX_LENGTH,
)

val_dataset = TextClassificationDataset(
    val_df["text"],
    val_df["label"],
    tokenizer,
    max_length=MAX_LENGTH,
)


# ----------------------------
# Fine-tune BERT
# ----------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

# have not toyed around with these hyperparameters yet. I was satisfied enough with the first iteration. These may be just fine as-is.
training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir=f"{MODEL_OUTPUT_DIR}/logs",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print(f"Saved fine-tuned model to {MODEL_OUTPUT_DIR}")

The results were good–almost too good... With just one epoch, the model was able to classify reports incredibly well. It achieved an F1-score of .974. And training loss was only .123 after one epoch. By three epochs, it was reaching an F1-Score of .985. Probably just needed two epochs. But after reviewing my code to make sure I wasn't mixing up labels somehow, and after reviewing the model's mistakes, I determined it was good. Turns out, BERT is quite capable of learning how to identify local coverage of racial violence.

Results:

Epoch: 1, 2, 3

Training Loss: 0.122600, 0.004100, 0.007900

Validation Loss: 0.059215, 0.062252, 0.061558

Accuracy: 0.982323, 0.984848, 0.989899

Precision: 0.970149, 0.984733, 0.984962

Recall: 0.977444, 0.969925, 0.984962

F1: 0.973783, 0.977273, 0.984962

Another thing: my process also lists the BERT_positive_probability for each classification. These probability labels are the model's more granular comparisons of the training data to the test data examples. They allow me to look at how certain the model is on its labels. Very useful.

## 4) Final BERT Training Version

This step was probly overkill, but just to give the model more examples, I decided to fine-tune BERT again but without splitting my training data by the 80/20 split. This meant I couldn't review this model's classifications with easily readable statistics, but given how well my first fine-tuning went, I was confident the model would do just fine.

This version, by the way, is the fine-tuned model I decided to deploy on my Cairo Bulletin data. By fine-tuning again, it would have 20% more examples to learn from.

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)


# ----------------------------
# Settings
# ----------------------------

MODEL_NAME = "bert-base-uncased"

# Use bert_training_data_chunked.csv here if you already chunked long training texts.
TRAIN_PATH = "test_data/bert_training_data.csv"

MODEL_OUTPUT_DIR = "bert_models/bert_local_violence_model_final"

BATCH_SIZE = 16
MAX_LENGTH = 512


# ----------------------------
# Device
# ----------------------------

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
print("MPS available:", torch.backends.mps.is_available())
print("MPS built:", torch.backends.mps.is_built())


# ----------------------------
# Dataset
# ----------------------------

class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(int(self.labels[idx]), dtype=torch.long),
        }


# ----------------------------
# Load all training data
# ----------------------------

df = pd.read_csv(TRAIN_PATH)

df = df[["text", "label"]].copy()
df["text"] = df["text"].fillna("").astype(str).str.strip()
df = df[df["text"] != ""]
df["label"] = df["label"].astype(int)

print("Training label counts:")
print(df["label"].value_counts())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = TextClassificationDataset(
    df["text"],
    df["label"],
    tokenizer,
    max_length=MAX_LENGTH,
)


# ----------------------------
# Fine-tune BERT on all labeled data
# ----------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

model.to(device)

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir=f"{MODEL_OUTPUT_DIR}/logs",
    logging_steps=25,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("Trainer device:", trainer.args.device)

trainer.train()

trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print(f"Saved final fine-tuned model to {MODEL_OUTPUT_DIR}")

## 5) Deploy BERT to Classify my Data

And now

In [ ]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# ----------------------------
# Settings for deployment
# ----------------------------

MODEL_OUTPUT_DIR = "bert_models/bert_local_violence_model_final"

TEST_PATH = "test_data/sn93055779_BERT_ready.csv"
TEMP_TEST_PATH = "test_data/sn93055779_BERT_ready.tmp.csv"

BATCH_SIZE = 16
MAX_LENGTH = 512
READ_CHUNK_SIZE = 2_000

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


# ----------------------------
# Classify test CSV and update it with BERT_label_01
# ----------------------------

model = AutoModelForSequenceClassification.from_pretrained(MODEL_OUTPUT_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_DIR)

model.to(device)
model.eval()

LABEL_COLUMN = "BERT_label_01"
PROB_COLUMN = "BERT_positive_probability"

first_write = True

for chunk_df in tqdm(
    pd.read_csv(TEST_PATH, chunksize=READ_CHUNK_SIZE),
    desc="Classifying rows",
):
    texts = chunk_df["text"].fillna("").astype(str).tolist()

    bert_labels = []
    bert_probs = []

    for start in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts[start:start + BATCH_SIZE]

        encoded = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)
            probs = torch.softmax(outputs.logits, dim=-1)

        positive_probs = probs[:, 1].detach().cpu()

        labels = (positive_probs >= 0.5).int().tolist()

        bert_probs.extend(positive_probs.tolist())
        bert_labels.extend(labels)

    chunk_df[LABEL_COLUMN] = bert_labels
    chunk_df[PROB_COLUMN] = bert_probs

    chunk_df.to_csv(
        TEMP_TEST_PATH,
        mode="w" if first_write else "a",
        header=first_write,
        index=False,
    )

    first_write = False

# Replace the original CSV only after the full classification succeeds.
os.replace(TEMP_TEST_PATH, TEST_PATH)

print(f"Updated {TEST_PATH} with columns: {LABEL_COLUMN}, {PROB_COLUMN}")

## 6) Some Review of the Results

Once it was done cooking, I reviewed results.

In [ ]:
import pandas as pd

df = pd.read_csv('test_data/sn93055779_BERT_ready.csv')

df['BERT_label_01'].value_counts()

The label counts were intriguing. And got me excited. Of the 90,431 chunks of text, the model labeled 1,374 of them as having local reports of racial violence. That's roughly 1.5% of the newspaper text. That's not a ton of examples, but given the wide range of subject matter in American newspapers at the time, I wouldn't anticipate local racial violence coverage would be much higher than that. And 1,374 examples are manageable in terms of hand-keying. In other words, I could reasonably review these results for accuracy myself.

Value Counts:

Rows NOT local racial violence: 89057
Rows with local racial violence: 1374

Let's pause a moment and reflect on how often I need to add the chron_am_url compiler to my data. Here it is again, right where I thought of it this time:

In [ ]:
seq_no = df['seq'].astype(str).str.extract(r'(\d+)', expand=False)

month = df['month'].astype(str).str.zfill(2)
day = df['day'].astype(str).str.zfill(2)

df['chron_am_url'] = (
    'https://www.loc.gov/resource/sn93055779/'
    + df['year'].astype(str)
    + '-'
    + month
    + '-'
    + day
    + '/'
    + df['ed'].astype(str)
    + '/?sp='
    + seq_no.fillna('')
)

df.to_csv('test_data/sn93055779_BERT_ready.csv')

Then I started reviewing the results by hand. In particular, I scrolled through the data arranged by the highest BERT_positive_probability labels first. I haven't read them carefully, but I did notice that the highest probability rows were in fact local coverage of violence, but not all were clearly racial violence. That was something I needed to determine more carefully in a hand-keying process. Here, I was just reading the ocr in BBEdit, copy-and-pasted from my dataframe. But more careful review would be in order.

In [ ]:
df

Then I decided to look for coverage of the event in question: the lynching of two Black men in Tamms, just north of Cairo, in September of 1913. To find coverage close to the date, I subset the data by year and month.

In [ ]:
matches = df[
    (df["year"] == 1913) &
    (df["month"] == 9)
]

matches

I arranged this subset by highest BERT_positive_probability label first. Then I perused those high probability rows, looking for any that were immediately after the date of the lynching: September 12th. This led me to notice one row from a page on September 13th with a BERT_probability_label of 94%. Promising!

I read the OCR and went to the full page. And omfg I think it worked! This helped me discover this article:

https://tile.loc.gov/image-services/iiif/service:ndnp:iune:batch_iune_kerning_ver01:data:sn93055779:00295872743:1913091301:0476/3657,4190,700,1594/full/0/default.jpg

It describes an event that seems to fit the example. Two Black men were accused of assaulting a Tamms saloon owner on Sept. 12th. This led to a white posse forming and chasing them down. At the time of this report, one of the Black men were shot to death. The other had not yet been apprehended.

Could this be the event in question? If so, the model found it.

I've got more reviewing to do, but as it currently stands, I think I'm working in the right direction.